# Grammar Nondeterminism Analysis

This notebook analyzes the grammar to determine:
1. How many sentences have multiple valid classes for the last word
2. Which grammar rules allow multiple valid classes at different positions
3. Statistics on ambiguity in the generated corpus


In [ ]:
import numpy as np
import json
import os
from collections import Counter, defaultdict
from nltk import CFG, ChartParser
from nltk.corpus import brown
from nltk.probability import FreqDist
from random import choice
import matplotlib.pyplot as plt

# Set random seed
np.random.seed(42)

print("Loading grammar and corpora...")


ModuleNotFoundError: No module named 'nltk'

In [ ]:
# Load saved corpora
corpora_dir = "corpora"

with open(os.path.join(corpora_dir, 'uniform_corpus.json'), 'r', encoding='utf-8') as f:
    uniform_corpus = json.load(f)

with open(os.path.join(corpora_dir, 'zipfian_corpus.json'), 'r', encoding='utf-8') as f:
    zipfian_corpus = json.load(f)

uniform_sentences = uniform_corpus['sentences']
uniform_classes = uniform_corpus['classes']
zipfian_sentences = zipfian_corpus['sentences']
zipfian_classes = zipfian_corpus['classes']

print(f"Loaded {len(uniform_sentences):,} uniform sentences")
print(f"Loaded {len(zipfian_sentences):,} zipfian sentences")


In [ ]:
# Recreate grammar (same as original notebook)
def clean_word(word):
    """Remove disambiguation suffixes like _1, _2"""
    if '_' in word:
        return word.rsplit('_', 1)[0]
    return word

# Build word lists from Brown corpus
tagged = brown.tagged_words(tagset='universal')
nouns = [w.lower() for w, t in tagged if t == 'NOUN']
verbs = [w.lower() for w, t in tagged if t == 'VERB']
dets  = [w.lower() for w, t in tagged if t == 'DET']
adjs  = [w.lower() for w, t in tagged if t == 'ADJ']
preps = [w.lower() for w, t in tagged if t == 'ADP']

def top_clean(words, n):
    fd = FreqDist(words)
    result = []
    for w, _ in fd.most_common():
        if not w.isalpha() or "'" in w or "\\" in w:
            continue
        result.append(w)
        if len(result) >= n:
            break
    return result

top_nouns = top_clean(nouns, 100)
top_verbs = top_clean(verbs, 100)
top_dets = top_clean(dets, 20)
top_adjs = top_clean(adjs, 50)
top_preps = top_clean(preps, 30)

# Create disambiguated word lists
disambig_nouns = [f"{w}(N)" for w in top_nouns]
disambig_verbs = [f"{w}(V)" for w in top_verbs]
disambig_dets = [f"{w}(Det)" for w in top_dets]
disambig_adjs = [f"{w}(Adj)" for w in top_adjs]
disambig_preps = [f"{w}(P)" for w in top_preps]

# Create grammar rules
noun_rules = [f"N -> '{w}'" for w in disambig_nouns]
verb_rules = [f"V -> '{w}'" for w in disambig_verbs]
det_rules = [f"Det -> '{w}'" for w in disambig_dets]
adj_rules = [f"Adj -> '{w}'" for w in disambig_adjs]
prep_rules = [f"P -> '{w}'" for w in disambig_preps]

# Word to category mapping
word2cats = {}
def add_mapping(words, cat):
    for w in words:
        word2cats.setdefault(clean_word(w), set()).add(cat)

add_mapping(disambig_nouns, 'N')
add_mapping(disambig_verbs, 'V')
add_mapping(disambig_dets,  'Det')
add_mapping(disambig_adjs,  'Adj')
add_mapping(disambig_preps, 'P')

# Base grammar
base_grammar = """
S -> NP VP
PP -> P NP
NP -> Det N | Det Adj N | Det N PP | Det Adj N PP | 'I'
VP -> V NP | VP PP
"""

full_grammar_str = (
    base_grammar
    + "\n"
    + "\n".join(noun_rules + verb_rules + det_rules + adj_rules + prep_rules)
)

grammar = CFG.fromstring(full_grammar_str)
parser = ChartParser(grammar)
gr = parser.grammar()

print("✓ Grammar loaded")


In [ ]:
# Method 1: Analyze grammar rules to find positions with multiple valid classes
print("="*60)
print("METHOD 1: ANALYZING GRAMMAR RULES")
print("="*60)

# For each production rule, check what can follow
# Focus on rules that can end sentences or lead to the last word

# Rules that can produce the last word in a sentence:
# - VP -> V NP (last word could be N from NP)
# - VP -> VP PP (last word could be N from PP -> P NP -> ... -> N)
# - NP -> Det N (last word is N)
# - NP -> Det Adj N (last word is N)
# - NP -> Det N PP (last word could be N from PP)
# - NP -> Det Adj N PP (last word could be N from PP)

# Check what can be the last word class
possible_last_classes = set()

# Direct: NP ending in N
possible_last_classes.add('N')

# From PP: PP -> P NP, and NP can end in N, so PP can end in N
# But PP itself is P, so if sentence ends with PP, last word is P
possible_last_classes.add('P')

# From VP -> V NP: if NP ends in N, last word is N
# From VP -> VP PP: if PP ends in N (from NP), last word is N
# But VP itself starts with V, so if sentence is just VP, first word is V

# Actually, let's think about sentence structure: S -> NP VP
# The last word comes from VP, which can be:
# - VP -> V NP: last word from NP (which can be N)
# - VP -> VP PP: last word from PP (which is P, or N if PP -> P NP -> ... -> N)

# So last word can be: N (from NP in VP), P (from PP), or potentially V if VP is just V?

# Let's check all productions that can lead to terminal symbols
terminal_productions = {}
for production in grammar.productions():
    lhs = str(production.lhs())
    if len(production.rhs()) == 1 and isinstance(production.rhs()[0], str):
        # Terminal production
        if lhs not in terminal_productions:
            terminal_productions[lhs] = []
        terminal_productions[lhs].append(production.rhs()[0])

print(f"\nTerminal productions by category:")
for cat, terms in terminal_productions.items():
    print(f"  {cat}: {len(terms)} words")

# Now check what categories can appear at the end of different structures
print(f"\nPossible last word classes from grammar analysis:")
print(f"  N: Yes (from NP)")
print(f"  P: Yes (from PP)")
print(f"  V: Possibly (if VP -> V with no NP)")
print(f"  Det: Unlikely (usually starts NP)")
print(f"  Adj: Unlikely (usually before N)")

# More systematic: generate many sentences and check what classes appear at the end
print(f"\n{'='*60}")
print("METHOD 2: EMPIRICAL ANALYSIS FROM GENERATED SENTENCES")
print(f"{'='*60}")


In [ ]:
# Analyze which contexts (first n-1 word CLASSES) allow multiple classes for the nth word
# We use class sequences, not words, since grammar nondeterminism is about class patterns

n_context = 5  # Same as training: use first 4 word classes to predict 5th

context_to_classes = defaultdict(set)  # Map class context -> set of possible next classes
context_counts = Counter()  # Count how many times each class context appears

print(f"Analyzing class contexts of length {n_context-1}...")
print(f"Checking if same class context can have different next word classes...")

for sentences, classes_list in [(uniform_sentences, uniform_classes), (zipfian_sentences, zipfian_classes)]:
    for sent, classes in zip(sentences, classes_list):
        if len(classes) >= n_context:
            # Get context as sequence of classes (first n-1 classes)
            context_classes = classes[:n_context-1]
            context_str = ' '.join(context_classes)  # Use class sequence, not words
            last_class = classes[n_context-1]
            
            context_to_classes[context_str].add(last_class)
            context_counts[context_str] += 1

print(f"\nTotal unique contexts: {len(context_to_classes):,}")
print(f"Total context occurrences: {sum(context_counts.values()):,}")

# Count how many contexts allow multiple classes
ambiguous_contexts = {ctx: classes for ctx, classes in context_to_classes.items() if len(classes) > 1}
unambiguous_contexts = {ctx: classes for ctx, classes in context_to_classes.items() if len(classes) == 1}

print(f"\nContexts with multiple possible classes: {len(ambiguous_contexts):,} ({100*len(ambiguous_contexts)/len(context_to_classes):.2f}%)")
print(f"Contexts with single possible class: {len(unambiguous_contexts):,} ({100*len(unambiguous_contexts)/len(context_to_classes):.2f}%)")

# Count sentences that have ambiguous contexts
ambiguous_sentence_count = 0
ambiguous_sentence_examples = []

for sentences, classes_list in [(uniform_sentences, uniform_classes), (zipfian_sentences, zipfian_classes)]:
    for sent, classes in zip(sentences, classes_list):
        if len(classes) >= n_context:
            context_classes = classes[:n_context-1]
            context_str = ' '.join(context_classes)
            if context_str in ambiguous_contexts:
                ambiguous_sentence_count += 1
                if len(ambiguous_sentence_examples) < 10:
                    # Store example with class sequence
                    ambiguous_sentence_examples.append((sent, context_str, context_to_classes[context_str], classes[:n_context]))

total_sentences = len(uniform_sentences) + len(zipfian_sentences)
print(f"\nSentences with ambiguous class contexts: {ambiguous_sentence_count:,} ({100*ambiguous_sentence_count/total_sentences:.2f}%)")

print(f"\nExample ambiguous class contexts (showing first 10):")
for i, (sent, ctx, possible_classes, full_context) in enumerate(ambiguous_sentence_examples[:10], 1):
    print(f"\n  {i}. Class context: '{ctx}'")
    print(f"     Possible next classes: {sorted(possible_classes)}")
    print(f"     Full sequence (first {n_context} classes): {' '.join(full_context)}")
    print(f"     Example sentence: {sent[:80]}...")


In [ ]:
# Detailed statistics on ambiguity
print(f"\n{'='*60}")
print("DETAILED AMBIGUITY STATISTICS")
print(f"{'='*60}")

# Distribution of number of possible classes per context
class_count_dist = Counter([len(classes) for classes in context_to_classes.values()])
print(f"\nDistribution of possible classes per context:")
for num_classes in sorted(class_count_dist.keys()):
    count = class_count_dist[num_classes]
    print(f"  {num_classes} class(es): {count:,} contexts ({100*count/len(context_to_classes):.2f}%)")

# Most ambiguous class contexts
most_ambiguous = sorted(ambiguous_contexts.items(), key=lambda x: len(x[1]), reverse=True)[:10]
print(f"\nMost ambiguous class contexts (top 10):")
for i, (ctx, classes) in enumerate(most_ambiguous, 1):
    print(f"  {i}. Class sequence '{ctx}' → {len(classes)} possible next classes: {sorted(classes)}")
    print(f"     Appears {context_counts[ctx]} times")

# Class combinations in ambiguous contexts (which sets of classes can follow the same context)
class_combinations = Counter([tuple(sorted(classes)) for classes in ambiguous_contexts.values()])
print(f"\nMost common class combinations in ambiguous contexts (top 10):")
print("(Shows which sets of classes can follow the same class context)")
for i, (combo, count) in enumerate(class_combinations.most_common(10), 1):
    print(f"  {i}. {combo}: {count:,} class contexts")


In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Distribution of number of classes per context
ax = axes[0]
class_counts = sorted(class_count_dist.keys())
counts = [class_count_dist[c] for c in class_counts]
ax.bar(class_counts, counts, alpha=0.7, color='steelblue')
ax.set_xlabel('Number of Possible Classes')
ax.set_ylabel('Number of Contexts')
ax.set_title('Distribution of Ambiguity\n(Number of Possible Classes per Class Context)')
ax.grid(True, alpha=0.3, axis='y')
for i, (c, cnt) in enumerate(zip(class_counts, counts)):
    ax.text(c, cnt, f'{cnt:,}', ha='center', va='bottom', fontsize=9)

# Plot 2: Pie chart of ambiguous vs unambiguous
ax = axes[1]
sizes = [len(unambiguous_contexts), len(ambiguous_contexts)]
labels = ['Unambiguous\n(1 class)', 'Ambiguous\n(>1 class)']
colors = ['lightgreen', 'lightcoral']
ax.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
ax.set_title('Class Context Ambiguity Distribution')

plt.tight_layout()
plt.savefig('grammar_ambiguity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPlot saved to: grammar_ambiguity_analysis.png")


In [ ]:
# Method 3: Analyze grammar rules directly to understand why ambiguity exists
print(f"\n{'='*60}")
print("METHOD 3: GRAMMAR RULE ANALYSIS")
print(f"{'='*60}")

# Check which non-terminals can produce which terminal categories
# This helps understand the source of ambiguity

def get_terminal_categories_at_end(grammar, non_terminal, max_depth=10, visited=None):
    """Recursively find all terminal categories that can appear at the END of a non-terminal"""
    if visited is None:
        visited = set()
    
    if non_terminal in visited or max_depth <= 0:
        return set()
    
    visited.add(non_terminal)
    categories = set()
    
    # Get all productions for this non-terminal
    productions = [p for p in grammar.productions() if str(p.lhs()) == non_terminal]
    
    for prod in productions:
        rhs = prod.rhs()
        if len(rhs) == 1:
            # Terminal production - this is the end
            if isinstance(rhs[0], str):
                # Extract category from word like "word(N)"
                if '(' in rhs[0] and ')' in rhs[0]:
                    cat = rhs[0].split('(')[1].rstrip(')')
                    categories.add(cat)
                else:
                    # Direct terminal, category is the LHS
                    categories.add(non_terminal)
        else:
            # Non-terminal production - check what can end the LAST symbol in RHS
            last_symbol = rhs[-1]
            if isinstance(last_symbol, str):
                # Last symbol is terminal
                if '(' in last_symbol and ')' in last_symbol:
                    cat = last_symbol.split('(')[1].rstrip(')')
                    categories.add(cat)
                else:
                    categories.add(non_terminal)
            else:
                # Last symbol is non-terminal - recurse
                symbol_str = str(last_symbol)
                sub_categories = get_terminal_categories_at_end(grammar, symbol_str, max_depth-1, visited.copy())
                categories.update(sub_categories)
    
    return categories

# Check what categories can end different structures
print(f"\nTerminal categories that can appear at the END:")
for nt in ['S', 'NP', 'VP', 'PP']:
    categories = get_terminal_categories_at_end(grammar, nt)
    print(f"  {nt} can end with: {sorted(categories)}")

# Since S -> NP VP, the last word comes from VP
# Check what VP can end with
vp_categories = get_terminal_categories_at_end(grammar, 'VP')
print(f"\nVP (which produces the last word in S -> NP VP) can end with: {sorted(vp_categories)}")
print(f"\nThis shows the grammar allows multiple classes at the sentence end position.")

# Summary
print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"1. Grammar allows multiple classes for the last word position")
print(f"2. {len(ambiguous_contexts):,} out of {len(context_to_classes):,} class contexts ({100*len(ambiguous_contexts)/len(context_to_classes):.2f}%) are ambiguous")
print(f"3. {ambiguous_sentence_count:,} out of {total_sentences:,} sentences ({100*ambiguous_sentence_count/total_sentences:.2f}%) have ambiguous class contexts")
print(f"4. This explains why the model may struggle - there is genuine class-level ambiguity in the grammar")
print(f"5. Analysis uses class sequences (not words) since grammar nondeterminism is about class patterns")
